This notebook simply makes a test for the creation of reactions in ChEBI format, instead of names. This will be implemented in 1+2/1+2.ipynb. It does obviously not work anymore, as transporters_df.tsv now is completed, with only the relevant columns. This notebook has fulfilled its goal, sort of speak.

In [1]:
import pandas as pd
import re
import numpy as np

First is the inital for the TCDB reactions\
Write this section below as a function as well, as done on the Rhea-part. Easier to integrate cleanly.

In [ ]:
df = pd.read_csv("../1+2/transporters_df.tsv", sep="\t", dtype=str)

name2chebi_dict = {}

with open("name2chebi.txt", "r") as file:
    for line in file:
        parts = line.strip().split(" ")
        if len(parts) == 2:
            name, chebi_id = parts
            name2chebi_dict[name] = chebi_id

def tcdb_convert_to_chebi(row):
    reaction = row["Reaction"]
    
    if pd.notna(reaction) and isinstance(reaction, str):

        reaction = re.sub(r"\s*\(in\)|\(out\)", "", reaction)

        chebi_name = row["CHEBI Name"]
        chebi_id = row["CHEBI ID"]
        
        if pd.notna(chebi_name) and pd.notna(chebi_id):
            reaction = reaction.replace(chebi_name, chebi_id)

        for name, chebi_id in name2chebi_dict.items():
            reaction = reaction.replace(name, chebi_id)
        
    return reaction

df["TCDB:Reaction:CHEBI"] = df.apply(tcdb_convert_to_chebi, axis=1)

The next step, is to create the reaction for Rhea as well, in ChEBI format. In R:ChEBI identifier, the ChEBIs appear in the order they appear in the reaction. That is handy.

In [ ]:
def rhea_convert_to_chebi(row):
    equation = str(row["R:Equation"]) if pd.notnull(row["R:Equation"]) else ""
    chebi_str = str(row["R:ChEBI identifier"]) if pd.notnull(row["R:ChEBI identifier"]) else ""

    # Avoid errors when empty
    if not equation.strip() or not chebi_str.strip():
        return np.nan

    chebis = [c.strip() for c in chebi_str.split(";")]

    # Clean equation text
    cleaned = re.sub(r"\((in|out)\)", "", equation)
    cleaned = re.sub(r"\(n\+1\)", "", cleaned)
    cleaned = re.sub(r"\(n\)", "", cleaned)
    cleaned = re.sub(r"\s+n\s+", " ", cleaned)
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()

    # Split on " + " and " = ", preserving operators
    tokens = re.split(r" ([+=]) ", cleaned)
    parts = [t.strip() for t in tokens if t.strip()]

    processed_names = []
    processed_ops = []

    i = 0
    while i < len(parts):
        token = parts[i]
        if token in ["+", "="]:
            processed_ops.append(f" {token} ")
        else:
            match = re.match(r"^(\d+)\s+(.+)$", token)
            if match:
                stoich, name = match.groups()
                processed_names.append((name, stoich))
            else:
                processed_names.append((token, ""))  # no stoichiometry
        i += 1

    # Map names to CHEBI IDs
    name_to_chebi = {}
    chebi_index = 0
    chebi_result = []
    
    for name, stoich in processed_names:
        if name not in name_to_chebi:
            if chebi_index < len(chebis):
                name_to_chebi[name] = chebis[chebi_index]
                chebi_index += 1
            else:
                name_to_chebi[name] = "MISSING_CHEBI" # Not an issue now, but kept in for later iterations
        chebi_id = name_to_chebi[name]
        chebi_result.append(f"{stoich} {chebi_id}".strip())

    # Reconstruct final reaction string
    result = [chebi_result[0]]
    for i, op in enumerate(processed_ops):
        result.append(op)
        result.append(chebi_result[i + 1])

    return "".join(result)

df["Rhea:Reaction:CHEBI"] = df.apply(rhea_convert_to_chebi, axis=1)